## GroupBy and Aggregation

In [44]:
import pandas as pd

In [46]:
data = {'Team': ['Marketing', 'Sales', 'Sales', 'HR', 'Marketing', 'HR'],
        'Employee': ['Alice', 'Bob', 'Charlie', 'David', 'Eve', 'Frank'],
        'Salary': [90000, 110000, 105000, 75000, 95000, 80000],
        'Projects': [3, 5, 4, 2, 4, 3]}

df = pd.DataFrame(data)

### 1. Splitting

The first step is splitting. Pandas takes the original DataFrame and partitions it into smaller DataFrames based on the criteria you provide in the by parameter. Each of these smaller DataFrames contains rows that share the same value for the specified key(s).

For our example df, if we group by the 'Team' column (df.groupby('Team')), pandas will create three groups: one for 'Marketing', one for 'Sales', and one for 'HR'. Internally, it creates a dictionary-like structure where the keys are the group names ('Marketing', 'Sales', 'HR') and the values are the row indices belonging to each group.

In [47]:
grouped = df.groupby('Team')
grouped.first()

,Employee,Salary,Projects
Team,,,
HR,David,75000,2
Marketing,Alice,90000,3
Sales,Bob,110000,5


### 2. Applying 
Next is the Applying phase. A function or operation is applied independently to each of the smaller DataFrames created in the splitting phase. This is where the actual computation happens. The operations fall into three main categories:

- Aggregation: Computes a summary statistic for each group, like the mean, sum, or count. This reduces the data to a single value per group. For example, calculating the average salary for each team.

- Transformation: Performs a group-specific calculation but returns a new series or DataFrame with the same shape as the original group. A common use case is standardizing data within a group (e.g., calculating a z-score). This is useful for feature engineering and is a core concept in reshaping data with pandas.

- Filtration: Removes entire groups based on a computed property. For example, you could keep only the teams with an average salary above a certain threshold.

### 3. Combining phase
Finally, the Combining phase takes the results from the applying phase and stitches them back together into a single pandas object (a DataFrame or Series). The structure of this final object depends on the operation you applied. 

For aggregations, you'll typically get a new DataFrame where the group keys form the index. For transformations, the output will have the same index as your original DataFrame. This final step presents the insights in a clean, organized format.

In [48]:
agg_result = grouped['Salary'].agg(['sum', 'mean'])
agg_result

,sum,mean
Team,,
HR,155000,77500.0
Marketing,185000,92500.0
Sales,215000,107500.0


## Key GroupBy Methods and Operations

### Aggregation methods

Aggregation methods summarize each group into a single value or set of values, making them ideal for computing statistics like totals, averages, or counts. Built-in functions such as `sum()`, `mean()`, `count()`, `min()`, `max()`, `std()`, and `var()` are optimized for performance and cover common use cases, such as calculating team performance metrics in sports data or sales totals by region in business reports.

Consider this sample DataFrame for demonstrations throughout this section:

In [49]:
import pandas as pd
df = pd.DataFrame({
    'team': ['A', 'A', 'B', 'B', 'C', 'C'],
    'points': [25, 12, 15, 14, 19, 23],
    'assists': [5, 7, 7, 9, 12, 9],
    'rebounds': [11, 8, 10, 6, 6, 5]
})

Using built-in aggregation:

In [50]:
grouped = df.groupby('team')
print(grouped.sum())

      points  assists  rebounds
team                           
A         37       12        19
B         29       16        16
C         42       21        11


For multiple and custom aggregations:

In [51]:
print(grouped.agg({
    'points': 'sum',
    'assists': 'mean',
    'rebounds': lambda x: x.max() - x.min()}))

      points  assists  rebounds
team                           
A         37      6.0         3
B         29      8.0         4
C         42     10.5         1


### Transformation methods

In [52]:
df['points_normalized'] = grouped['points'].transform(lambda x: (x - x.mean()) / x.std())
print(df)

  team  points  assists  rebounds  points_normalized
0    A      25        5        11           0.707107
1    A      12        7         8          -0.707107
2    B      15        7        10           0.707107
3    B      14        9         6          -0.707107
4    C      19       12         6          -0.707107
5    C      23        9         5           0.707107


### Filtration  methods  

In [53]:
filtered = grouped.filter(lambda x: x['points'].sum() > 30)
print(filtered)

  team  points  assists  rebounds  points_normalized
0    A      25        5        11           0.707107
1    A      12        7         8          -0.707107
4    C      19       12         6          -0.707107
5    C      23        9         5           0.707107


## Other examples

In [54]:
data = {
    'product': ['Laptop', 'Phone', 'Laptop', 'Tablet', 'Phone', 'Tablet'],
    'sales': [1200, 800, 1500, 600, 900, 700],
    'region': ['East', 'West', 'East', 'West', 'East', 'West']
}
df = pd.DataFrame(data)

Grouping by a single column

In [55]:
# Group by 'product' and aggregate sales with sum and mean
grouped = df.groupby('product')
agg_result = grouped['sales'].agg(['sum', 'mean'])
print(agg_result)

          sum    mean
product              
Laptop   2700  1350.0
Phone    1700   850.0
Tablet   1300   650.0


Grouping by multiple columns

In [56]:
multi_grouped = df.groupby(['product', 'region'])
multi_aggregation = multi_grouped['sales'].sum()
print(multi_aggregation)

product  region
Laptop   East      2700
Phone    East       900
         West       800
Tablet   West      1300
Name: sales, dtype: int64


Aggregating multiple columns

In [57]:
df['units'] = [5, 10, 6, 8, 12, 9]

# Group by 'product' and apply different aggregations
agg_multi_col = df.groupby('product').agg({
    'sales': 'sum',
    'units': 'mean'
})
print(agg_multi_col)

         sales  units
product              
Laptop    2700    5.5
Phone     1700   11.0
Tablet    1300    8.5
